<a href="https://colab.research.google.com/github/KrishnaKarthikReddy/DLL/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NLP ASSIGNMENT-2**

# KORPOLE KRISHNA KARTHIK REDDY

# 160123737186

# IT-3

# 1. Build a machine translation system using LSTM.(write code ). CO4 BL6

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# 1. Sample Dataset (English → Telugu)
# -------------------------
data = [
    ("i am happy", "నేను సంతోషంగా ఉన్నాను"),
    ("i am sad", "నేను బాధగా ఉన్నాను"),
    ("he is good", "అతను మంచివాడు"),
    ("she is kind", "ఆమె దయగలది"),
    ("they are here", "వారు ఇక్కడ ఉన్నారు")
]

# -------------------------
# 2. Tokenization
# -------------------------
def tokenize(sentence):
    return sentence.lower().split()

SRC_vocab = {"<pad>":0, "<sos>":1, "<eos>":2}
TRG_vocab = {"<pad>":0, "<sos>":1, "<eos>":2}

def build_vocab(data):
    for src, trg in data:
        for word in tokenize(src):
            if word not in SRC_vocab:
                SRC_vocab[word] = len(SRC_vocab)
        for word in tokenize(trg):
            if word not in TRG_vocab:
                TRG_vocab[word] = len(TRG_vocab)

build_vocab(data)

SRC_ivocab = {v:k for k,v in SRC_vocab.items()}
TRG_ivocab = {v:k for k,v in TRG_vocab.items()}

def sentence_to_tensor(sentence, vocab):
    tokens = tokenize(sentence)
    ids = [vocab["<sos>"]] + [vocab[t] for t in tokens] + [vocab["<eos>"]]
    return torch.tensor(ids, dtype=torch.long, device=device)

# -------------------------
# 3. Model
# -------------------------
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim)

    def forward(self, src):
        embedded = self.embedding(src).unsqueeze(1)  # (seq_len,1,emb_dim)
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(0).unsqueeze(1)
        embedded = self.embedding(input)

        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))

        prediction = self.fc(output.squeeze(0).squeeze(0))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        hidden, cell = self.encoder(src)

        trg_len = trg.shape[0]
        output_dim = len(TRG_vocab)

        outputs = torch.zeros(trg_len, output_dim).to(device)

        input = trg[0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output

            top1 = output.argmax(0)
            input = trg[t] if random.random() < teacher_forcing_ratio else top1

        return outputs

# -------------------------
# 4. Initialize
# -------------------------
INPUT_DIM = len(SRC_vocab)
OUTPUT_DIM = len(TRG_vocab)
EMB_DIM = 32
HIDDEN_DIM = 64

encoder = Encoder(INPUT_DIM, EMB_DIM, HIDDEN_DIM).to(device)
decoder = Decoder(OUTPUT_DIM, EMB_DIM, HIDDEN_DIM).to(device)
model = Seq2Seq(encoder, decoder).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=TRG_vocab["<pad>"])

# -------------------------
# 5. Training
# -------------------------
EPOCHS = 300

for epoch in range(EPOCHS):
    total_loss = 0

    for src_sentence, trg_sentence in data:
        src_tensor = sentence_to_tensor(src_sentence, SRC_vocab)
        trg_tensor = sentence_to_tensor(trg_sentence, TRG_vocab)

        optimizer.zero_grad()

        output = model(src_tensor, trg_tensor)

        output_dim = output.shape[1]

        output = output[1:].view(-1, output_dim)
        trg = trg_tensor[1:].view(-1)

        loss = criterion(output, trg)
        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

# -------------------------
# 6. Translate
# -------------------------
def translate(sentence, model, max_len=10):
    model.eval()

    src_tensor = sentence_to_tensor(sentence, SRC_vocab)
    hidden, cell = model.encoder(src_tensor)

    input = torch.tensor(TRG_vocab["<sos>"], device=device)

    result = []

    for _ in range(max_len):
        output, hidden, cell = model.decoder(input, hidden, cell)
        pred_token = output.argmax(0).item()

        if pred_token == TRG_vocab["<eos>"]:
            break

        result.append(TRG_ivocab[pred_token])
        input = torch.tensor(pred_token, device=device)

    return " ".join(result)

# -------------------------
# 7. Test
# -------------------------
print("\nTranslations:")
for src, _ in data:
    print(f"{src} → {translate(src, model)}")

Epoch 0, Loss: 13.3805
Epoch 50, Loss: 0.0082
Epoch 100, Loss: 0.0029
Epoch 150, Loss: 0.0015
Epoch 200, Loss: 0.0009
Epoch 250, Loss: 0.0006

Translations:
i am happy → నేను సంతోషంగా ఉన్నాను
i am sad → నేను బాధగా ఉన్నాను
he is good → అతను మంచివాడు
she is kind → ఆమె దయగలది
they are here → వారు ఇక్కడ ఉన్నారు


# 2. Evaluate Convolutional neural networks for sentence classification (write code) CO4 BL5.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# 1. Dataset (Larger + Better)
# -------------------------
data = [
    ("i love this movie", 1),
    ("this film is great", 1),
    ("amazing experience", 1),
    ("i enjoyed the story", 1),
    ("fantastic acting", 1),

    ("i hate this movie", 0),
    ("this film is terrible", 0),
    ("worst acting ever", 0),
    ("i did not like this", 0),
    ("very bad experience", 0)
]

random.shuffle(data)

# Split dataset
train_data = data[:8]
test_data = data[8:]

# -------------------------
# 2. Tokenization & Vocabulary
# -------------------------
def tokenize(text):
    return text.lower().split()

vocab = {"<pad>":0}

for sentence, _ in train_data:
    for word in tokenize(sentence):
        if word not in vocab:
            vocab[word] = len(vocab)

def encode(sentence):
    return [vocab.get(word, 0) for word in tokenize(sentence)]

max_len = max(len(tokenize(s)) for s, _ in train_data)

def pad(seq):
    return seq + [0]*(max_len - len(seq))

def prepare_dataset(dataset):
    X, y = [], []
    for s, label in dataset:
        X.append(pad(encode(s)))
        y.append(label)
    return torch.tensor(X, dtype=torch.long).to(device), torch.tensor(y).to(device)

X_train, y_train = prepare_dataset(train_data)
X_test, y_test = prepare_dataset(test_data)

# -------------------------
# 3. CNN Model
# -------------------------
class TextCNN(nn.Module):
    def __init__(self, vocab_size, emb_dim, num_filters, filter_sizes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)

        self.convs = nn.ModuleList([
            nn.Conv2d(1, num_filters, (fs, emb_dim))
            for fs in filter_sizes
        ])

        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(len(filter_sizes)*num_filters, 2)

    def forward(self, x):
        x = self.embedding(x)              # (batch, seq_len, emb_dim)
        x = x.unsqueeze(1)                # (batch, 1, seq_len, emb_dim)

        convs = [torch.relu(conv(x)).squeeze(3) for conv in self.convs]
        pools = [torch.max(c, dim=2)[0] for c in convs]

        out = torch.cat(pools, dim=1)
        out = self.dropout(out)
        return self.fc(out)

# -------------------------
# 4. Initialize
# -------------------------
model = TextCNN(len(vocab), emb_dim=50, num_filters=10, filter_sizes=[2,3,4]).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# -------------------------
# 5. Training
# -------------------------
EPOCHS = 100

for epoch in range(EPOCHS):
    model.train()

    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# -------------------------
# 6. Evaluation (ON UNSEEN DATA)
# -------------------------
model.eval()

with torch.no_grad():
    preds = model(X_test).argmax(dim=1)

accuracy = (preds == y_test).float().mean().item()

TP = ((preds==1) & (y_test==1)).sum().item()
FP = ((preds==1) & (y_test==0)).sum().item()
FN = ((preds==0) & (y_test==1)).sum().item()

precision = TP / (TP + FP + 1e-8)
recall = TP / (TP + FN + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

print("\nEvaluation (Test Set):")
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

# -------------------------
# 7. Test on NEW SENTENCES
# -------------------------
def predict(sentence):
    model.eval()
    seq = pad(encode(sentence))
    tensor = torch.tensor([seq], dtype=torch.long).to(device)

    with torch.no_grad():
        output = model(tensor)
        pred = output.argmax(1).item()

    return "Positive" if pred == 1 else "Negative"

print("\nCustom Predictions:")
test_sentences = [
    "i really loved this",
    "this was not good",
    "amazing acting",
    "worst movie ever"
]

for s in test_sentences:
    print(f"{s} → {predict(s)}")

Epoch 0, Loss: 0.5143
Epoch 20, Loss: 0.0023
Epoch 40, Loss: 0.0005
Epoch 60, Loss: 0.0000
Epoch 80, Loss: 0.0007

Evaluation (Test Set):
Accuracy: 0.50
Precision: 0.50
Recall: 1.00
F1 Score: 0.67

Custom Predictions:
i really loved this → Positive
this was not good → Positive
amazing acting → Positive
worst movie ever → Positive


# 3. Implement-Translate English sentences to French using pretrained models.(write code) CO5 BL4.

In [ ]:
# -------------------------
# 0. Install (run once in Colab/terminal)
# -------------------------
# !pip install transformers sentencepiece torch

from transformers import MarianMTModel, MarianTokenizer
import torch

# -------------------------
# 1. Load Pretrained Model
# -------------------------
model_name = "Helsinki-NLP/opus-mt-en-fr"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)

# -------------------------
# 2. Single Sentence Translation
# -------------------------
def translate(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# -------------------------
# 3. Batch Translation
# -------------------------
def translate_batch(sentences):
    inputs = tokenizer(sentences, return_tensors="pt", padding=True).to(device)
    outputs = model.generate(**inputs)
    translations = [tokenizer.decode(t, skip_special_tokens=True) for t in outputs]
    return translations

# -------------------------
# 4. Test Data
# -------------------------
sentences = [
    "I am happy",
    "This is a beautiful place",
    "I love learning artificial intelligence",
    "The weather is very good today",
    "He is not a good person"
]

# -------------------------
# 5. Run Batch Translation
# -------------------------
print("\nTranslations (English → French):\n")

translations = translate_batch(sentences)

for s, t in zip(sentences, translations):
    print(f"{s} → {t}")

# -------------------------
# 6. Test Single Translation
# -------------------------
print("\nSingle Sentence Test:\n")
print("Input: I am learning deep learning")
print("Output:", translate("I am learning deep learning"))

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Translations (English → French):

I am happy → Je suis heureux.
This is a beautiful place → C'est un bel endroit.
I love learning artificial intelligence → J'aime apprendre l'intelligence artificielle
The weather is very good today → Le temps est très bon aujourd'hui
He is not a good person → Il n'est pas quelqu'un de bien.

Single Sentence Test:

Input: I am learning deep learning
Output: J'apprends l'apprentissage profond


# 4. Implement Question-answering based systems.(write code) CO5 BL4

In [ ]:
from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2"
)

context = """
Artificial Intelligence (AI) is the simulation of human intelligence in machines.
These machines can learn from experience, adapt to new inputs, and perform tasks
such as problem-solving, decision-making, and language understanding.
AI is widely used in industries including healthcare, finance, education, and transportation.
"""

questions = [
    "What is Artificial Intelligence?",
    "Where is AI used?",
    "What tasks can AI perform?",
    "What does AI learn from?"
]

print("\nFinal QA System:\n")

for q in questions:
    result = qa_pipeline(question=q, context=context)

    answer = result['answer']
    score = result['score']

    print(f"Q: {q}")

    if score < 0.3:
        print("A: Answer not confident")
    else:
        print(f"A: {answer}")

    print(f"Confidence: {score:.4f}")
    print("-"*50)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Final QA System:

Q: What is Artificial Intelligence?
A: the simulation of human intelligence in machines
Confidence: 0.4608
--------------------------------------------------
Q: Where is AI used?
A: healthcare, finance, education, and transportation
Confidence: 0.5978
--------------------------------------------------
Q: What tasks can AI perform?
A: problem-solving, decision-making, and language understanding
Confidence: 0.9199
--------------------------------------------------
Q: What does AI learn from?
A: experience
Confidence: 0.9576
--------------------------------------------------


# 5. Develop  the code part to Generate text from a prompt. .(write code ) CO5 BL6

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# -------------------------
# Load Model
# -------------------------
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Fix padding
tokenizer.pad_token = tokenizer.eos_token

# -------------------------
# Generate Function
# -------------------------
def generate_text(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],  # ✅ FIX
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        num_return_sequences=2,
        pad_token_id=tokenizer.eos_token_id
    )

    results = []
    for output in outputs:
        text = tokenizer.decode(output, skip_special_tokens=True)
        results.append(text)

    return results

# -------------------------
# Test
# -------------------------
prompt = "Artificial Intelligence is transforming the world because"

outputs = generate_text(prompt)

print("\nFinal Clean Output (No Warnings):\n")

for i, text in enumerate(outputs):
    print(f"Output {i+1}:")
    print(text)
    print("-"*50)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Final Clean Output (No Warnings):

Output 1:
Artificial Intelligence is transforming the world because it has a huge potential for transformative change. In this post we will take you on an interesting journey through how artificial intelligence can transform society and provide opportunities to develop new technologies, create better jobs (including healthcare), build innovative solutions that are sustainable as well improving our quality of life by enabling people to enjoy greater freedom at every level regardless if they're unemployed or not in need
The World Economic Forum's 2015 Global Entrepreneurship Report outlines four key areas where AI could help drive innovation:

"AI might be able offer insights into human behavior; but more importantly…It may also open up possibilities beyond
--------------------------------------------------
Output 2:
Artificial Intelligence is transforming the world because it's changing our lives. It enables us to live in a very different way from how

# 6. Build Text summarization techniques in nlp  .(write code ) C05 BL6

In [ ]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6"
)

text = """
Artificial Intelligence (AI) is transforming industries such as healthcare,
finance, and transportation. It enables machines to learn from data and perform
tasks that typically require human intelligence. AI improves efficiency,
automation, and decision-making. However, it also raises concerns about ethics,
privacy, and job displacement. Researchers are working to make AI systems more
transparent, fair, and reliable for future applications.
"""

summary = summarizer(
    text,
    max_length=90,
    min_length=30,
    length_penalty=1.5,
    do_sample=False
)

print("\nSummary:\n")
print(summary[0]['summary_text'])


Summary:

 Artificial Intelligence (AI) is transforming industries such as healthcare, finance, and transportation . It enables machines to learn from data and perform tasks that typically require human intelligence .


# 7. Implement Topic Modeling Concept in NLP. (write code ) CO5 BL5

In [ ]:
# -------------------------
# 0. Install (run once if needed)
# -------------------------
# !pip install nltk scikit-learn

import nltk
import re
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

nltk.download('stopwords')

# -------------------------
# 1. Dataset
# -------------------------
documents = [
    # AI
    "Artificial intelligence machine learning data science",
    "Machine learning models analyze data prediction",
    "Deep learning neural networks artificial intelligence",
    "Data science statistics machine learning algorithms",
    "AI models process datasets prediction analysis",

    # Sports
    "Football basketball cricket sports fans",
    "Basketball players sports training performance",
    "Cricket football sports matches fans",
    "Sports competitions tournaments athletes",
    "Athletes sports training competitions performance"
]

# -------------------------
# 2. Preprocessing
# -------------------------
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

clean_docs = [preprocess(doc) for doc in documents]

# -------------------------
# 3. Vectorization
# -------------------------
vectorizer = CountVectorizer(
    stop_words='english',
    ngram_range=(1,2),
    max_df=0.85
)

X = vectorizer.fit_transform(clean_docs)

# -------------------------
# 4. LDA Model
# -------------------------
lda = LatentDirichletAllocation(
    n_components=2,
    max_iter=40,
    random_state=42
)

lda.fit(X)

# -------------------------
# 5. Extract Topics
# -------------------------
feature_names = vectorizer.get_feature_names_out()

print("\nDiscovered Topics:")
topic_words_map = {}

for i, topic in enumerate(lda.components_):
    words = [feature_names[j] for j in topic.argsort()[-10:]]
    topic_words_map[i] = words
    print(f"\nTopic {i}: {words}")

# -------------------------
# 6. Auto Topic Labeling (Robust)
# -------------------------
AI_KEYWORDS = {"ai", "machine", "learning", "data", "neural", "intelligence", "models", "prediction"}
SPORTS_KEYWORDS = {"sports", "football", "basketball", "cricket", "athletes", "fans", "players", "training"}

topic_labels = {}

for topic_id, words in topic_words_map.items():
    ai_score = sum(1 for w in words if w in AI_KEYWORDS)
    sports_score = sum(1 for w in words if w in SPORTS_KEYWORDS)

    if ai_score > sports_score:
        topic_labels[topic_id] = "AI / Machine Learning"
    else:
        topic_labels[topic_id] = "Sports"

print("\nAuto-detected Topic Labels:", topic_labels)

# -------------------------
# 7. Topic Distribution
# -------------------------
topic_dist = lda.transform(X)

# -------------------------
# 8. Smart Classification Fix
# -------------------------
def smart_assign(doc, probs, topic_labels):
    doc_lower = doc.lower()

    # 1) Strong SPORTS anchors (highest priority)
    if any(w in doc_lower for w in ["football", "basketball", "cricket", "sports", "athletes", "fans", "players"]):
        return "Sports"

    # 2) Strong AI anchors
    if any(w in doc_lower for w in ["ai", "machine learning", "data science", "neural", "datasets", "prediction", "models"]):
        return "AI / Machine Learning"

    # 3) Fallback to LDA probability
    topic_id = probs.argmax()
    return topic_labels[topic_id]

# -------------------------
# 9. Final Output
# -------------------------
print("\nFinal Document Classification:\n")

for i, doc in enumerate(documents):
    probs = topic_dist[i]
    label = smart_assign(doc, probs, topic_labels)

    print(doc)
    print(f"Assigned Topic: {label}")
    print(f"Probabilities: {probs}")
    print("-"*50)


Discovered Topics:

Topic 0: ['models process', 'machine learning', 'machine', 'data', 'artificial intelligence', 'artificial', 'intelligence', 'models', 'prediction', 'learning']

Topic 1: ['athletes', 'competitions', 'fans', 'cricket', 'football', 'basketball', 'training', 'performance', 'sports training', 'sports']

Auto-detected Topic Labels: {0: 'AI / Machine Learning', 1: 'Sports'}

Final Document Classification:

Artificial intelligence machine learning data science
Assigned Topic: AI / Machine Learning
Probabilities: [0.07272056 0.92727944]
--------------------------------------------------
Machine learning models analyze data prediction
Assigned Topic: AI / Machine Learning
Probabilities: [0.95026148 0.04973852]
--------------------------------------------------
Deep learning neural networks artificial intelligence
Assigned Topic: AI / Machine Learning
Probabilities: [0.95413311 0.04586689]
--------------------------------------------------
Data science statistics machine lea

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# 8. Build Simple Transformer Encoder Block using PyTorch .(write code ) CO4 BL6

In [ ]:
import torch
import torch.nn as nn

# -------------------------
# 1. Transformer Encoder Block
# -------------------------
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super(TransformerEncoderBlock, self).__init__()

        # Multi-head self-attention
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)

        # Layer Normalization
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        # Feed Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )

        # Dropout
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (seq_len, batch_size, embed_dim)

        # ---- Self Attention ----
        attn_output, _ = self.attention(x, x, x)

        # Add & Norm
        x = self.norm1(x + self.dropout(attn_output))

        # ---- Feed Forward ----
        ffn_output = self.ffn(x)

        # Add & Norm
        x = self.norm2(x + self.dropout(ffn_output))

        return x

# -------------------------
# 2. Test the Encoder Block
# -------------------------
if __name__ == "__main__":
    embed_dim = 64
    num_heads = 4
    ff_dim = 128
    seq_len = 10
    batch_size = 2

    # Create random input
    x = torch.rand(seq_len, batch_size, embed_dim)

    # Initialize encoder
    encoder = TransformerEncoderBlock(embed_dim, num_heads, ff_dim)

    # Forward pass
    output = encoder(x)

    print("Input shape :", x.shape)
    print("Output shape:", output.shape)

Input shape : torch.Size([10, 2, 64])
Output shape: torch.Size([10, 2, 64])


In [ ]:
print(torch.allclose(x, output))

False


In [ ]:
x = torch.rand(10, 2, 64)
output = encoder(x)

print(output.shape)  # should be same as input


torch.Size([10, 2, 64])


In [ ]:
print(torch.allclose(x, output))

False


In [ ]:
x = torch.zeros(10, 2, 64)
output = encoder(x)

print(output)

tensor([[[ 1.3680,  0.0888, -0.8736,  ..., -1.4636, -0.4286, -1.6477],
         [ 0.4009,  0.2635, -0.7768,  ..., -1.4146, -0.2958, -1.6135]],

        [[ 1.3079,  0.0869, -0.8318,  ..., -1.3949, -0.4070, -1.5706],
         [ 1.3709,  0.1370, -0.7914,  ..., -1.3605, -0.3621, -1.5380]],

        [[ 1.4357,  0.1199, -0.8701,  ..., -1.4770, -0.4123,  0.2506],
         [ 1.2891,  0.0517, -0.8793,  ..., -1.4500, -0.4488, -1.6281]],

        ...,

        [[ 1.3729,  0.2323, -0.8463,  ..., -1.4303,  0.2323, -1.6125],
         [ 1.4836,  0.1711, -0.8164,  ..., -1.4218, -0.3598, -1.6106]],

        [[ 1.2916,  0.0253, -0.9274,  ..., -1.5115, -0.4869, -1.6937],
         [ 1.3475,  0.1058, -0.8285,  ..., -1.4012, -0.3965, -1.5798]],

        [[ 1.3938,  0.1207, -0.8372,  ..., -1.4244, -0.3943, -1.6076],
         [ 0.2522,  0.1295, -0.8001,  ..., -1.3700, -0.3703, -1.5477]]],
       grad_fn=<NativeLayerNormBackward0>)


# **9. Build a text classification model using Naive Bayes.(write code ) CO3 BL6**

In [4]:
# ==========================================================
# TEXT CLASSIFICATION USING NAIVE BAYES
# Example: Spam vs Ham Classification
# ==========================================================

# Import libraries
import pandas as pd

# Scikit-learn modules
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==========================================================
# STEP 1: Create Dataset
# ==========================================================
# label -> 1 = Spam, 0 = Ham

data = {
    'message': [
        'Congratulations you won a lottery claim now',
        'Free entry in a prize competition',
        'Call now to claim your reward',
        'Hey are we meeting today',
        'Please send me the notes',
        'Let us go for lunch tomorrow',
        'Win cash prize now',
        'Important update about your account',
        'Exclusive offer just for you',
        'Can you help me with homework',
        'Claim your free vacation now',
        'Meeting postponed to tomorrow',
        'Win a free mobile phone',
        'Submit your assignment today',
        'Special discount offer available'
    ],

    'label': [
        1, 1, 1, 0, 0,
        0, 1, 0, 1, 0,
        1, 0, 1, 0, 1
    ]
}

# Convert to DataFrame
df = pd.DataFrame(data)

# Display dataset
print("====================================")
print("DATASET")
print("====================================")
print(df)

# ==========================================================
# STEP 2: Split Data into Train and Test
# ==========================================================
# stratify=y ensures balanced class distribution

X = df['message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,        # 30% test data
    random_state=42,
    stratify=y            # Important for balanced classes
)

print("\n====================================")
print("TRAINING DATA SIZE:", len(X_train))
print("TEST DATA SIZE:", len(X_test))
print("====================================")

# ==========================================================
# STEP 3: Convert Text to Numerical Features
# ==========================================================
# CountVectorizer creates Bag-of-Words model

vectorizer = CountVectorizer()

# Learn vocabulary from training data
X_train_features = vectorizer.fit_transform(X_train)

# Transform test data using same vocabulary
X_test_features = vectorizer.transform(X_test)

# Display vocabulary
print("\nVocabulary Learned:")
print(vectorizer.get_feature_names_out())

# ==========================================================
# STEP 4: Train Naive Bayes Model
# ==========================================================
model = MultinomialNB()

# Train model
model.fit(X_train_features, y_train)

# ==========================================================
# STEP 5: Predict on Test Data
# ==========================================================
y_pred = model.predict(X_test_features)

# ==========================================================
# STEP 6: Model Evaluation
# ==========================================================
print("\n====================================")
print("MODEL EVALUATION")
print("====================================")

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

# Confusion Matrix
print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

# ==========================================================
# STEP 7: Show Actual vs Predicted
# ==========================================================
print("\n====================================")
print("ACTUAL vs PREDICTED")
print("====================================")

for message, actual, predicted in zip(X_test, y_test, y_pred):
    actual_label = "Spam" if actual == 1 else "Ham"
    predicted_label = "Spam" if predicted == 1 else "Ham"

    print(f"Message: {message}")
    print(f"Actual: {actual_label}")
    print(f"Predicted: {predicted_label}")
    print("------------------------------------")

# ==========================================================
# STEP 8: Predict New Custom Messages
# ==========================================================
new_messages = [
    "Congratulations! You won free tickets",
    "Are you coming to college today",
    "Exclusive discount claim now",
    "Please attend class tomorrow"
]

# Convert messages to features
new_features = vectorizer.transform(new_messages)

# Predict
predictions = model.predict(new_features)

print("\n====================================")
print("CUSTOM MESSAGE PREDICTIONS")
print("====================================")

for msg, pred in zip(new_messages, predictions):
    if pred == 1:
        print(f"'{msg}' --> Spam")
    else:
        print(f"'{msg}' --> Ham")

# ==========================================================
# STEP 9: Display Probabilities (Optional)
# ==========================================================
print("\n====================================")
print("PREDICTION PROBABILITIES")
print("====================================")

probabilities = model.predict_proba(new_features)

for msg, prob in zip(new_messages, probabilities):
    print(f"Message: {msg}")
    print(f"Ham Probability : {prob[0]:.4f}")
    print(f"Spam Probability: {prob[1]:.4f}")
    print("------------------------------------")

DATASET
                                        message  label
0   Congratulations you won a lottery claim now      1
1             Free entry in a prize competition      1
2                 Call now to claim your reward      1
3                      Hey are we meeting today      0
4                      Please send me the notes      0
5                  Let us go for lunch tomorrow      0
6                            Win cash prize now      1
7           Important update about your account      0
8                  Exclusive offer just for you      1
9                 Can you help me with homework      0
10                 Claim your free vacation now      1
11                Meeting postponed to tomorrow      0
12                      Win a free mobile phone      1
13                 Submit your assignment today      0
14             Special discount offer available      1

TRAINING DATA SIZE: 10
TEST DATA SIZE: 5

Vocabulary Learned:
['are' 'call' 'can' 'cash' 'claim' 'congratulatio

# 10.Implement text classification using Logistic Regression. (write code ) CO3 BL4

In [3]:
# ==========================================================
# TEXT CLASSIFICATION USING LOGISTIC REGRESSION
# Example: Spam vs Ham Message Classification
# ==========================================================

# Import libraries
import pandas as pd

# Scikit-learn modules
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==========================================================
# STEP 1: Create Dataset
# ==========================================================
# label -> 1 = Spam, 0 = Ham

data = {
    'message': [
        'Congratulations you won a lottery claim now',
        'Free entry in a prize competition',
        'Call now to claim your reward',
        'Hey are we meeting today',
        'Please send me the notes',
        'Let us go for lunch tomorrow',
        'Win cash prize now',
        'Important update about your account',
        'Exclusive offer just for you',
        'Can you help me with homework',
        'Claim your free vacation now',
        'Meeting postponed to tomorrow',
        'Win a free mobile phone',
        'Submit your assignment today',
        'Special discount offer available',
        'Limited time free bonus reward',
        'Project meeting at 5 PM',
        'Please review the document',
        'Earn money instantly now',
        'Join us for dinner tonight'
    ],

    'label': [
        1, 1, 1, 0, 0,
        0, 1, 0, 1, 0,
        1, 0, 1, 0, 1,
        1, 0, 0, 1, 0
    ]
}

# Convert to DataFrame
df = pd.DataFrame(data)

# Display dataset
print("====================================")
print("DATASET")
print("====================================")
print(df)

# ==========================================================
# STEP 2: Split Data into Train and Test
# ==========================================================
X = df['message']
y = df['label']

# stratify keeps class distribution balanced
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("\n====================================")
print("TRAINING DATA SIZE:", len(X_train))
print("TEST DATA SIZE:", len(X_test))
print("====================================")

# ==========================================================
# STEP 3: Convert Text into Numerical Features
# ==========================================================
# TF-IDF gives better weighting than simple word count

vectorizer = TfidfVectorizer(stop_words='english')

# Learn vocabulary from training set
X_train_features = vectorizer.fit_transform(X_train)

# Transform test set
X_test_features = vectorizer.transform(X_test)

print("\nVocabulary Learned:")
print(vectorizer.get_feature_names_out())

# ==========================================================
# STEP 4: Train Logistic Regression Model
# ==========================================================
# max_iter increased for convergence

model = LogisticRegression(max_iter=1000)

# Train model
model.fit(X_train_features, y_train)

# ==========================================================
# STEP 5: Predictions
# ==========================================================
y_pred = model.predict(X_test_features)

# ==========================================================
# STEP 6: Model Evaluation
# ==========================================================
print("\n====================================")
print("MODEL EVALUATION")
print("====================================")

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

# Confusion Matrix
print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

# ==========================================================
# STEP 7: Actual vs Predicted
# ==========================================================
print("\n====================================")
print("ACTUAL vs PREDICTED")
print("====================================")

for message, actual, predicted in zip(X_test, y_test, y_pred):
    actual_label = "Spam" if actual == 1 else "Ham"
    predicted_label = "Spam" if predicted == 1 else "Ham"

    print(f"Message: {message}")
    print(f"Actual: {actual_label}")
    print(f"Predicted: {predicted_label}")
    print("------------------------------------")

# ==========================================================
# STEP 8: Predict New Custom Messages
# ==========================================================
new_messages = [
    "Congratulations! You won free tickets",
    "Are you coming to college today",
    "Exclusive discount claim now",
    "Please attend class tomorrow",
    "Earn free cash reward instantly"
]

# Convert to TF-IDF features
new_features = vectorizer.transform(new_messages)

# Predict
predictions = model.predict(new_features)

print("\n====================================")
print("CUSTOM MESSAGE PREDICTIONS")
print("====================================")

for msg, pred in zip(new_messages, predictions):
    if pred == 1:
        print(f"'{msg}' --> Spam")
    else:
        print(f"'{msg}' --> Ham")

# ==========================================================
# STEP 9: Prediction Probabilities
# ==========================================================
print("\n====================================")
print("PREDICTION PROBABILITIES")
print("====================================")

probabilities = model.predict_proba(new_features)

for msg, prob in zip(new_messages, probabilities):
    print(f"Message: {msg}")
    print(f"Ham Probability : {prob[0]:.4f}")
    print(f"Spam Probability: {prob[1]:.4f}")
    print("------------------------------------")

# ==========================================================
# STEP 10: Important Features (Top Spam Indicators)
# ==========================================================
# Logistic Regression coefficients show word importance

feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]

# Top 10 spam words
top_spam_indices = coefficients.argsort()[-10:]

print("\n====================================")
print("TOP WORDS INDICATING SPAM")
print("====================================")

for index in reversed(top_spam_indices):
    print(f"{feature_names[index]} : {coefficients[index]:.4f}")

DATASET
                                        message  label
0   Congratulations you won a lottery claim now      1
1             Free entry in a prize competition      1
2                 Call now to claim your reward      1
3                      Hey are we meeting today      0
4                      Please send me the notes      0
5                  Let us go for lunch tomorrow      0
6                            Win cash prize now      1
7           Important update about your account      0
8                  Exclusive offer just for you      1
9                 Can you help me with homework      0
10                 Claim your free vacation now      1
11                Meeting postponed to tomorrow      0
12                      Win a free mobile phone      1
13                 Submit your assignment today      0
14             Special discount offer available      1
15               Limited time free bonus reward      1
16                      Project meeting at 5 PM      0
17